### Week 5 Day 4

AutoGen Core - Distributed

I'm only going to give a Teaser of this!!

Partly because I'm unsure how relevant it is to you. If you'd like me to add more content for this, please do let me know..

In [12]:
from dataclasses import dataclass
import asyncio
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = True

### Start with our Message class

In [13]:

@dataclass
class Message:
    content: str

### And now - a host for our distributed runtime

In [14]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start()

### Let's reintroduce a tool

In [15]:
serper = GoogleSerperAPIWrapper()
langchain_serper = Tool(name="internet_serper", func=serper.run, description="Use for a single web search when strictly needed")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [16]:
instruction1 = "You are preparing a quick decision brief. Give 4 concise bullets for why to choose AutoGen. Use prior knowledge first; if needed, call internet_serper at most once. Keep total response under 120 words."

instruction2 = "You are preparing a quick decision brief. Give 4 concise bullets for why NOT to choose AutoGen. Use prior knowledge first; if needed, call internet_serper at most once. Keep total response under 120 words."

judge = "You must decide whether to use AutoGen. You will receive pros and cons from your team. Base your answer only on that input and return: 1) Decision, 2) 2-sentence rationale."

### And make some Agents

In [17]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(
            name,
            model_client=model_client,
            tools=[autogen_serper],
            reflect_on_tool_use=False,
            system_message="Be concise. Use at most one tool call. Return only final bullets.",
        )

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(
            name,
            model_client=model_client,
            tools=[autogen_serper],
            reflect_on_tool_use=False,
            system_message="Be concise. Use at most one tool call. Return only final bullets.",
        )

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)


In [18]:

from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:
    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:

    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    
    agent_id = AgentId("judge", "default")

In [19]:
response = await asyncio.wait_for(
    worker.send_message(Message(content="GO!"), agent_id),
    timeout=45,
    )

In [20]:
display(Markdown(response.content))

## Pros of AutoGen:
Because AutoGen is used by Microsoft and with its backing lots of usage scenarios are going through it. Plus, it has no revenue pressure as it's funded by MS. Naturally handles ambiguity, feedback, progress, and collaboration. · Enables effective coding-related tasks, like tool use with back-and-forth ... Its strength is its flexibility for solving complex, exploratory problems . It shines in R&D, autonomous software development, and deep data ... While AutoGen offers several advantages in data processing, such as speed and efficiency, it's important to recognize its limitations. AutoGen is a groundbreaking tool that significantly simplifies developing applications with GPT-4. It efficiently manages the intricate details. Microsoft AutoGen is an open-source framework for building AI agents and other artificial intelligence applications. Gagan Bansal, Senior Researcher, Microsoft Research AI Frontiers introduces a transformative update to the AutoGen framework that builds on user ... AutoGen is a new project from Microsoft Research that allows developers to build multi-agent teams to accomplish almost any task. Missing: benefits software. AutoGen simplifies the process of building an AI system that can collaborate, reason, and solve complex problems through agent-to-agent interactions.
A framework for building AI agents and applications · Deterministic and dynamic agentic workflows for business processes. · Research on multi-agent collaboration. Missing: features | Show results with:features. Key Features. Asynchronous messaging: Agents communicate through asynchronous messages, supporting both event-driven and request/response interaction patterns. One of the most exciting features of AutoGen is its ability to generate, execute, and verify code automatically using conversational AI agents. It offers features such as agents that can converse with other agents, LLM and tool use support, autonomous and human-in-the-loop workflows, and multi-agent ... AutoGen is a framework for creating multi-agent AI applications that can act autonomously or work alongside humans. Microsoft AutoGen is an open-source framework for building AI agents and other artificial intelligence applications. Learn how Microsoft AutoGen enables AI agents to collaborate, execute code, and integrate with Azure—driving real business value beyond the ... AutoGen's Layered Architecture. Microsoft Autogen features a layered architecture (Core, AgentChat & Extensions), a design that provides ... AutoGen is a framework for building multi-agent applications - and we recently released a new v0.4 version - a complete rewrite of the framework. AutoGen is a framework that helps you easily create multi-agent applications. Multi-agent applications are a relatively recent idea that involve defining ... Missing: features | Show results with:features.

## Cons of AutoGen:
inconsistent outputs; harder debugging; rising costs; fragile behavior. Once I stopped forcing agents into places they didn't belong, everything ... Missing: disadvantages | Show results with:disadvantages. autogens main problems are that its documentation is quite hard to read, with not enough examples, and there are somethings that flat out don't ... Missing: disadvantages | Show results with:disadvantages. ... AutoGen workflow: 11:20 Other AutoGen limitations: 33 ... AutoGen workflow: 11:20 Other AutoGen limitations: 33:10. Is AutoGen just HYPE ... Missing: disadvantages | Show results with:disadvantages. Compare Autogen vs LangChain vs CrewAI in this in-depth guide. Discover features, pros and cons, and the best use cases for your AI agent ... It scales better for parallel agents, but the downside is potential verbosity agents might over-discuss, inflating token usage by 30-50% in chat ... Explore AutoGen's capabilities and limitations in multi-agent workflows, distinguishing hype from reality. Gain insights into its potential and current ... Missing: disadvantages | Show results with:disadvantages. While AutoGen offers powerful capabilities, it requires some level of coding expertise, potentially limiting accessibility for non-technical users. The platform ... Missing: disadvantages | Show results with:disadvantages. 1. LLMs are Bad at Reasoning · 2. Multi-agent Workflows are Expensive · 3. You Will Quickly Run Into Rate Limits · 4. Working with Open-source LLMs ... Missing: disadvantages | Show results with:disadvantages. 1. Learning Curve. Users find the learning curve challenging, often feeling frustrated while trying to master AutogenAI's capabilities. · 2. User Difficulty · 3.



## Decision:

1) Decision: Use AutoGen.

2) Rationale: Despite its inconsistencies and the challenges of debugging, the benefits of AutoGen, such as strong backing from Microsoft, its ability to streamline complex problem-solving, and its efficiency in multi-agent collaboration, present significant advantages. The flexibility and innovative features it offers for developing AI applications make it a valuable tool, especially in research and development contexts. 

TERMINATE

In [21]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [22]:
await host.stop()